# 08 - Demo: Batch Ensemble Predictions

This notebook runs the ensemble on input texts and writes predictions from the
- LoRA Gemma meta-model

By default it reads the quick-fix CSV generated by notebook 09: `tm_research/ensemble/artifacts/demo_test_30_predictions_with_probs.csv`.
Set `USE_TEST_SPLIT = True` in the config cell if you want to load the processed test split directly (`tm_research/data/processed/test_final.csv`) and resolve probabilities via text lookup.

It is inference-only and reuses existing artifacts under `tm_research/ensemble/artifacts/`.

**Google Colab:** run the `%pip` cell first. Put the repo on Drive and set `REPO_ROOT` in the next cell if your path differs from `/content/drive/MyDrive/thesis/topicmodeling`. Add an `HF_TOKEN` secret (Gemma-2 license accepted) so `setup_colab()` can load the base model; artifacts are copied from the persistent `artifacts/` tree on Drive into `/content` scratch. If you see `OSError: [Errno 107] Transport endpoint is not connected`, the Drive mount dropped: **Runtime → Restart session**, remount Drive, and re-run from the top.

In [1]:
%pip install -q -U 'torchao>=0.16.0'
%pip install -q 'transformers>=4.44' 'peft>=0.11' 'bitsandbytes>=0.43' accelerate datasets pandas scikit-learn

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 105.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 45.2 MB/s eta 0:00:00


In [2]:
from __future__ import annotations

import json
import os
import sys
from pathlib import Path
from typing import Dict, List

REPO_ROOT = '/content/drive/MyDrive/thesis/topicmodeling/tm_research'
TMP_ROOT = '/content/ensemble_tmp'
REPO_PATH = Path(REPO_ROOT)
IMPORT_ROOT = str(REPO_PATH.parent if REPO_PATH.name == 'tm_research' else REPO_PATH)

if 'google.colab' in sys.modules:
    from google.colab import drive
    if not os.path.ismount('/content/drive'):
        drive.mount('/content/drive')
    if IMPORT_ROOT not in sys.path:
        # append: avoid importlib probing Drive before site-packages (errno 107 on flaky FUSE)
        sys.path.append(IMPORT_ROOT)
else:
    import pathlib
    LOCAL = pathlib.Path.cwd().resolve()
    while LOCAL.name != 'tm_research' and LOCAL.parent != LOCAL:
        LOCAL = LOCAL.parent
    repo_parent = str(LOCAL.parent)
    if repo_parent not in sys.path:
        sys.path.append(repo_parent)

# Import stack deps before tm_research so Drive is not first on sys.path during importlib.
import numpy as np
import pandas as pd
import torch

from tm_research.ensemble.colab_setup import setup_colab
paths = setup_colab(repo_root=REPO_ROOT, tmp_root=TMP_ROOT)

from tm_research.ensemble.utils_io import DEFAULT_DATA_DIR, TEST_FILE, load_label_map
from tm_research.ensemble.utils_stacking import (
    BASE_MODEL_NAMES,
    format_prompt,
    parse_label_from_completion,
)

# Derive repo artifacts dir directly from REPO_PATH — immune to env var / import order issues.
REPO_ARTIFACTS_DIR = REPO_PATH / 'ensemble' / 'artifacts'

print('Repo artifacts dir:', REPO_ARTIFACTS_DIR)
print('device =', 'cuda' if torch.cuda.is_available() else 'cpu')

Mounted at /content/drive
[utils_io] pulled 75 files: /content/drive/MyDrive/thesis/topicmodeling/tm_research/ensemble/artifacts -> /content/ensemble_tmp/artifacts
[colab_setup] is_colab=True
[colab_setup] repo_root=/content/drive/MyDrive/thesis/topicmodeling/tm_research
[colab_setup] tmp_root=/content/ensemble_tmp (hf_cache/bert_work/lora_work)
[colab_setup] artifacts=/content/ensemble_tmp/artifacts
[colab_setup] persistent_artifacts=/content/drive/MyDrive/thesis/topicmodeling/tm_research/ensemble/artifacts
[colab_setup] HF_TOKEN: set
Repo artifacts dir: /content/drive/MyDrive/thesis/topicmodeling/tm_research/ensemble/artifacts
device = cuda


In [3]:
# Config
# Default: consume notebook 09 output (already sampled 30% test with per-model probs).
USE_TEST_SPLIT = False
TEST_CSV = DEFAULT_DATA_DIR / TEST_FILE  # tm_research/data/processed/test_final.csv

INPUT_CSV = REPO_ARTIFACTS_DIR / 'demo_test_30_predictions_with_probs.csv'
OUTPUT_CSV = REPO_ARTIFACTS_DIR / 'demo_predictions.csv'

TEXT_COL = 'text'
MAX_NEW_TOKENS = 12
LORA_BATCH_SIZE = 4
DEMO_DATA_FRACTION = 1.0  # keep 1.0 to avoid re-sampling notebook 09's 30% subset
DEMO_SAMPLE_SEED = 42

# If input CSV does not provide probability columns, fallback to lookup by text
# from known split CSVs + saved base-model probs.
ENABLE_TEXT_LOOKUP_FALLBACK = True

In [4]:
# Load label map, weights, and LoRA metadata
label_map = load_label_map()

weights_path = REPO_ARTIFACTS_DIR / 'weights.json'
if not weights_path.exists():
    raise FileNotFoundError(f'Missing required artifact: {weights_path}')
with open(weights_path, 'r', encoding='utf-8') as f:
    weights_payload = json.load(f)

weights = weights_payload.get('weights', weights_payload)
missing_weight_models = [m for m in BASE_MODEL_NAMES if m not in weights]
if missing_weight_models:
    raise ValueError(f'Missing weights for models: {missing_weight_models}')

lora_meta_path = REPO_ARTIFACTS_DIR / 'lora_train_meta.json'
if not lora_meta_path.exists():
    raise FileNotFoundError(f'Missing required artifact: {lora_meta_path}')
with open(lora_meta_path, 'r', encoding='utf-8') as f:
    lora_meta = json.load(f)

LORA_DIR = REPO_ARTIFACTS_DIR / 'lora_adapter'
if not LORA_DIR.exists():
    raise FileNotFoundError(f'Missing required artifact dir: {LORA_DIR}')

# 06_train_meta_lora_gemma saves base_model; older JSON may use model_name
MODEL_NAME = lora_meta.get('base_model') or lora_meta.get('model_name')
if not MODEL_NAME:
    raise KeyError(
        "lora_train_meta.json must include 'base_model' (see 06_train_meta_lora_gemma) or legacy 'model_name'"
    )
from tm_research.ensemble.utils_stacking import build_meta_system_prompt
if lora_meta.get('system_prompt'):
    SYSTEM_PROMPT = lora_meta['system_prompt']
elif weights_payload.get('system_prompt'):
    SYSTEM_PROMPT = weights_payload['system_prompt']
else:
    SYSTEM_PROMPT = build_meta_system_prompt(label_map, weights)
LORA_INFER_MAX_LEN = int(lora_meta.get('max_seq_len', 800))

print('Loaded class names:', label_map.class_names)
print('Loaded weights:', {k: round(float(v), 4) for k, v in weights.items()})
print('LoRA base model:', MODEL_NAME)
print('LORA_INFER_MAX_LEN:', LORA_INFER_MAX_LEN)

Loaded class names: ['Anger', 'Disgust', 'Enjoyment', 'Fear', 'Other', 'Sadness', 'Surprise']
Loaded weights: {'phobert': 0.205, 'cafebert': 0.2107, 'vibert': 0.2003, 'logreg': 0.1923, 'svc': 0.1918}
LoRA base model: google/gemma-2-9b-it
LORA_INFER_MAX_LEN: 800


In [5]:
# Helpers for probability sources

def _prob_col_name(model_name: str, label_name: str) -> str:
    return f'{model_name}__{label_name}'


def get_probs_from_columns(input_df: pd.DataFrame) -> Dict[str, np.ndarray] | None:
    probs: Dict[str, np.ndarray] = {}
    n = len(input_df)
    for model_name in BASE_MODEL_NAMES:
        cols = [_prob_col_name(model_name, lab) for lab in label_map.class_names]
        if not all(c in input_df.columns for c in cols):
            return None
        arr = input_df[cols].to_numpy(dtype=np.float32)
        if arr.shape != (n, label_map.num_classes):
            raise ValueError(f'Unexpected shape for {model_name}: {arr.shape}')
        probs[model_name] = arr
    return probs


def build_lookup_probs() -> Dict[str, Dict[str, np.ndarray]]:
    from tm_research.ensemble.utils_io import DEFAULT_DATA_DIR, load_probs

    split_to_file = {
        'train': DEFAULT_DATA_DIR / 'train_1500_final.csv',
        'val': DEFAULT_DATA_DIR / 'val_final.csv',
        'test': DEFAULT_DATA_DIR / 'test_final.csv',
    }
    split_to_prob_key = {'train': 'oof', 'val': 'val', 'test': 'test'}

    lookup = {m: {} for m in BASE_MODEL_NAMES}
    for split, csv_path in split_to_file.items():
        if not csv_path.exists():
            continue
        raw = pd.read_csv(csv_path)
        text_col = 'Sentence_clean' if 'Sentence_clean' in raw.columns else ('Sentence' if 'Sentence' in raw.columns else 'text')
        texts = raw[text_col].astype(str).str.strip().tolist()
        prob_key = split_to_prob_key[split]
        for m in BASE_MODEL_NAMES:
            p = load_probs(m, prob_key)
            if len(p) != len(texts):
                raise ValueError(f'Length mismatch for {m}/{split}: probs={len(p)} texts={len(texts)}')
            for i, t in enumerate(texts):
                if t and t not in lookup[m]:
                    lookup[m][t] = p[i]
    return lookup


def get_probs_from_lookup(input_df: pd.DataFrame, lookup: Dict[str, Dict[str, np.ndarray]]) -> Dict[str, np.ndarray]:
    n = len(input_df)
    out = {}
    texts = input_df[TEXT_COL].astype(str).str.strip().tolist()
    for m in BASE_MODEL_NAMES:
        rows = []
        missing = []
        for i, txt in enumerate(texts):
            vec = lookup[m].get(txt)
            if vec is None:
                missing.append(i)
                rows.append(np.zeros(label_map.num_classes, dtype=np.float32))
            else:
                rows.append(np.asarray(vec, dtype=np.float32))
        if missing:
            raise ValueError(
                f'Missing lookup probabilities for model={m} at row indices {missing[:10]}'
                + (' ...' if len(missing) > 10 else '')
                + '. Provide per-model probability columns in input CSV for arbitrary new texts.'
            )
        out[m] = np.vstack(rows)
    return out

In [6]:
# Load input: default = full test split (for ensemble + optional gold metrics)
if USE_TEST_SPLIT:
    if not TEST_CSV.exists():
        raise FileNotFoundError(f'Test CSV not found: {TEST_CSV}')
    raw = pd.read_csv(TEST_CSV)
    src_col = (
        'Sentence_clean'
        if 'Sentence_clean' in raw.columns
        else ('Sentence' if 'Sentence' in raw.columns else 'text')
    )
    input_df = pd.DataFrame({TEXT_COL: raw[src_col].astype(str)})
    if 'Emotion' in raw.columns:
        input_df['gold_label'] = raw['Emotion'].astype(str).str.strip()
    elif 'label' in raw.columns:
        input_df['gold_label'] = raw['label'].astype(str).str.strip()
    print('Data source: test split', TEST_CSV)
else:
    if not Path(INPUT_CSV).exists():
        raise FileNotFoundError(
            f'Input CSV not found: {INPUT_CSV}. Set USE_TEST_SPLIT=True or create a CSV with `{TEXT_COL}`.'
        )
    input_df = pd.read_csv(INPUT_CSV)
    if TEXT_COL not in input_df.columns:
        raise ValueError(f'Input CSV must include `{TEXT_COL}` column. Columns={list(input_df.columns)}')
    print('Data source: custom CSV', INPUT_CSV)

input_df[TEXT_COL] = input_df[TEXT_COL].astype(str).str.strip()
input_df = input_df[input_df[TEXT_COL] != ''].reset_index(drop=True)
if input_df.empty:
    raise ValueError('No non-empty text rows.')

_n_full = len(input_df)
if DEMO_DATA_FRACTION < 1.0:
    k = max(1, int(round(_n_full * float(DEMO_DATA_FRACTION))))
    input_df = input_df.sample(n=k, random_state=DEMO_SAMPLE_SEED).reset_index(drop=True)
    print(f'Sampled {len(input_df)}/{_n_full} rows (DEMO_DATA_FRACTION={DEMO_DATA_FRACTION}, seed={DEMO_SAMPLE_SEED}).')

print('Input rows:', len(input_df))
if 'gold_label' in input_df.columns:
    print('Sample (text, gold_label):')
    print(input_df[[TEXT_COL, 'gold_label']].head(5).to_string(index=False))
else:
    print(input_df.head(3).to_string(index=False))

Data source: custom CSV /content/drive/MyDrive/thesis/topicmodeling/tm_research/ensemble/artifacts/demo_test_30_predictions_with_probs.csv
Input rows: 208
Sample (text, gold_label):
                                                                                                                                 text gold_label
      đôi khi thích nói chuyện chỉ vì nói chuyện hợp và vì thích nói chuyện chứ thật ra chả phải yêu đương hay thích kiểu trai gái gì      Other
                                                                                                   đơn giản vậy thôi là đủ yêu thương  Enjoyment
                                                                                                  câu hỏi thú vị vui vẻ vui vẻ vui vẻ  Enjoyment
lương này là tính luôn quá trình tham nhũng ăn bớt ăn xén rồi đó ! tụi mày chưa đủ trình đó thì lấy tư cách gì nói người ta nói xạo ?    Disgust
                                                                                             

In [7]:
# Resolve per-model probabilities for each input row
probs_per_model = get_probs_from_columns(input_df)

if probs_per_model is None:
    if not ENABLE_TEXT_LOOKUP_FALLBACK:
        raise ValueError(
            'Input CSV is missing probability columns and text-lookup fallback is disabled.'
        )
    print('Probability columns not found. Trying text-lookup fallback from known split artifacts...')
    lookup = build_lookup_probs()
    probs_per_model = get_probs_from_lookup(input_df, lookup)
else:
    print('Using probabilities from input CSV columns.')

for m in BASE_MODEL_NAMES:
    print(m, probs_per_model[m].shape)

Using probabilities from input CSV columns.
phobert (208, 7)
cafebert (208, 7)
vibert (208, 7)
logreg (208, 7)
svc (208, 7)


In [8]:
# Initialize result dataframe
result_df = input_df.copy()
print('Input rows ready for meta-model:', len(result_df))
print(result_df[[TEXT_COL]].head(5).to_string(index=False))

Weighted predictions ready: 208
                                                                                                                                 text pred_weighted  conf_weighted
      đôi khi thích nói chuyện chỉ vì nói chuyện hợp và vì thích nói chuyện chứ thật ra chả phải yêu đương hay thích kiểu trai gái gì         Other       0.432925
                                                                                                   đơn giản vậy thôi là đủ yêu thương     Enjoyment       0.807441
                                                                                                  câu hỏi thú vị vui vẻ vui vẻ vui vẻ     Enjoyment       0.769792
lương này là tính luôn quá trình tham nhũng ăn bớt ăn xén rồi đó ! tụi mày chưa đủ trình đó thì lấy tư cách gì nói người ta nói xạo ?         Anger       0.505909
                                                                                                      quá đơn rản , bạn thân yêu ơi ,       Sadness      

In [9]:
# Load LoRA meta-model (QLoRA-style 4-bit on GPU, same pattern as 07_evaluate_ensemble)
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from peft import PeftModel

if not os.environ.get('HF_TOKEN'):
    raise RuntimeError(
        'Set HF_TOKEN with Gemma-2 license accepted. On Colab: Secrets → HF_TOKEN, grant this notebook access, then re-run the setup cell.'
    )

bnb = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type='nf4',
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_use_double_quant=True,
)
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, token=os.environ['HF_TOKEN'])
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

base_model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    quantization_config=bnb,
    device_map='auto',
    torch_dtype=torch.bfloat16,
    attn_implementation='eager',
    token=os.environ['HF_TOKEN'],
)
model_lora = PeftModel.from_pretrained(base_model, str(LORA_DIR))
model_lora.eval()

print('LoRA model loaded from', LORA_DIR)

config.json:   0%|          | 0.00/857 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/47.0k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/17.5M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/636 [00:00<?, ?B/s]

[transformers] `torch_dtype` is deprecated! Use `dtype` instead!


...

Fetching 4 files:   0%|          | 0/4 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/464 [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/bitsandbytes/backends/cuda/ops.py:213: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)


generation_config.json:   0%|          | 0.00/173 [00:00<?, ?B/s]

LoRA model loaded from /content/drive/MyDrive/thesis/topicmodeling/tm_research/ensemble/artifacts/lora_adapter


In [10]:
# LoRA prompt building and generation

def render_chat(prompt_block: str) -> str:
    return tokenizer.apply_chat_template(
        [{'role': 'user', 'content': SYSTEM_PROMPT + '\n\n' + prompt_block}],
        tokenize=False,
        add_generation_prompt=True,
    )


def generate_labels(rows: List[dict], batch_size: int = 4, max_new_tokens: int = 12) -> tuple[list[str], list[str]]:
    preds: List[str] = []
    raws: List[str] = []
    for i in range(0, len(rows), batch_size):
        chunk = rows[i:i + batch_size]
        chats = [render_chat(r['prompt']) for r in chunk]
        enc = tokenizer(
            chats,
            return_tensors='pt',
            padding=True,
            truncation=True,
            max_length=2048,
        ).to(model_lora.device)

        with torch.inference_mode():
            out = model_lora.generate(
                **enc,
                max_new_tokens=max_new_tokens,
                do_sample=False,
                temperature=0.0,
                pad_token_id=tokenizer.pad_token_id,
            )

        gen_tokens = out[:, enc['input_ids'].shape[1]:]
        texts = tokenizer.batch_decode(gen_tokens, skip_special_tokens=True)
        for t in texts:
            raws.append(t)
            lab = parse_label_from_completion(t, label_map)
            preds.append(lab if lab is not None else 'UNKNOWN')
    return preds, raws


def build_meta_rows(input_texts: List[str], probs_dict: Dict[str, np.ndarray]) -> List[dict]:
    rows: List[dict] = []
    for i, txt in enumerate(input_texts):
        per_model = {m: probs_dict[m][i] for m in BASE_MODEL_NAMES}
        formatted = format_prompt(
            text=txt,
            probs_per_model=per_model,
            weights=weights,
            label_map=label_map,
            label=None,
        )
        rows.append({'prompt': formatted['prompt']})
    return rows

In [14]:
# Run LoRA meta-model predictions
meta_rows = build_meta_rows(result_df[TEXT_COL].tolist(), probs_per_model)
pred_lora, raw_lora = generate_labels(
    meta_rows,
    batch_size=LORA_BATCH_SIZE,
    max_new_tokens=MAX_NEW_TOKENS,
)

result_df['pred_lora'] = pred_lora
result_df['raw_lora_completion'] = raw_lora

print(result_df[[TEXT_COL, 'pred_lora']].head(15).to_string(index=False))

if 'gold_label' in result_df.columns:
    from sklearn.metrics import accuracy_score, f1_score

    yt = np.array([label_map.label2id[str(lab)] for lab in result_df['gold_label']])
    mask_l = np.array([lab in label_map.label2id for lab in result_df['pred_lora']])
    if mask_l.any():
        y_l = np.array([label_map.label2id[str(lab)] for lab in result_df.loc[mask_l, 'pred_lora']])
        yt_l = yt[mask_l]
        print(f'\nTest accuracy: LoRA={accuracy_score(yt_l, y_l):.4f}')
        print(
            f'Test macro-F1: LoRA={f1_score(yt_l, y_l, average="macro", zero_division=0):.4f}'
        )
    else:
        print('  LoRA=(no valid labels)')
    unk = int((~mask_l).sum())
    if unk:
        print(f'(LoRA UNKNOWN on {unk} rows)')

_display_cols = [
    TEXT_COL,
    'pred_lora',
    'raw_lora_completion',
]
if 'gold_label' in result_df.columns:
    _display_cols = [
        TEXT_COL,
        'gold_label',
        'pred_lora',
        'raw_lora_completion',
    ]
_n = len(result_df)
MAX_INLINE_DISPLAY = 400
print(f'\n=== Predictions for display ({_n} rows) ===')
if _n <= MAX_INLINE_DISPLAY:
    with pd.option_context('display.max_rows', None, 'display.max_colwidth', 100, 'display.width', None):
        print(result_df[_display_cols].to_string(index=True))
else:
    print(f'(Too many rows to print fully; first 200 of {_n}. Full table: {OUTPUT_CSV})')
    with pd.option_context('display.max_rows', None, 'display.max_colwidth', 100, 'display.width', None):
        print(result_df[_display_cols].head(200).to_string(index=True))

                                                                                                                                 text pred_weighted pred_lora
      đôi khi thích nói chuyện chỉ vì nói chuyện hợp và vì thích nói chuyện chứ thật ra chả phải yêu đương hay thích kiểu trai gái gì         Other     Other
                                                                                                   đơn giản vậy thôi là đủ yêu thương     Enjoyment Enjoyment
                                                                                                  câu hỏi thú vị vui vẻ vui vẻ vui vẻ     Enjoyment Enjoyment
lương này là tính luôn quá trình tham nhũng ăn bớt ăn xén rồi đó ! tụi mày chưa đủ trình đó thì lấy tư cách gì nói người ta nói xạo ?         Anger     Anger
                                                                                                      quá đơn rản , bạn thân yêu ơi ,       Sadness   Sadness
                                                    

In [12]:
# Save output CSV
OUTPUT_CSV.parent.mkdir(parents=True, exist_ok=True)
result_df.to_csv(OUTPUT_CSV, index=False, encoding='utf-8')
print('Saved predictions to:', OUTPUT_CSV)
print('Columns:', list(result_df.columns))

Saved predictions to: /content/drive/MyDrive/thesis/topicmodeling/tm_research/ensemble/artifacts/demo_predictions.csv
Columns: ['orig_idx', 'text', 'gold_label', 'pred_weighted', 'conf_weighted', 'phobert__Anger', 'phobert__Disgust', 'phobert__Enjoyment', 'phobert__Fear', 'phobert__Other', 'phobert__Sadness', 'phobert__Surprise', 'cafebert__Anger', 'cafebert__Disgust', 'cafebert__Enjoyment', 'cafebert__Fear', 'cafebert__Other', 'cafebert__Sadness', 'cafebert__Surprise', 'vibert__Anger', 'vibert__Disgust', 'vibert__Enjoyment', 'vibert__Fear', 'vibert__Other', 'vibert__Sadness', 'vibert__Surprise', 'logreg__Anger', 'logreg__Disgust', 'logreg__Enjoyment', 'logreg__Fear', 'logreg__Other', 'logreg__Sadness', 'logreg__Surprise', 'svc__Anger', 'svc__Disgust', 'svc__Enjoyment', 'svc__Fear', 'svc__Other', 'svc__Sadness', 'svc__Surprise', 'weighted__Anger', 'weighted__Disgust', 'weighted__Enjoyment', 'weighted__Fear', 'weighted__Other', 'weighted__Sadness', 'weighted__Surprise', 'pred_lora', 'ra

In [13]:
# Optional: tiny custom CSV when not using the test split
if USE_TEST_SPLIT:
    print('USE_TEST_SPLIT=True; smoke-test CSV creator skipped.')
elif not Path(INPUT_CSV).exists():
    test_src = pd.read_csv(DEFAULT_DATA_DIR / TEST_FILE)
    txt_col = 'Sentence_clean' if 'Sentence_clean' in test_src.columns else ('Sentence' if 'Sentence' in test_src.columns else 'text')
    demo = pd.DataFrame({TEXT_COL: test_src[txt_col].astype(str).head(3)})
    demo.to_csv(INPUT_CSV, index=False, encoding='utf-8')
    print('Created smoke-test input at', INPUT_CSV)
else:
    print('INPUT_CSV already exists; smoke-test creator skipped.')

INPUT_CSV already exists; smoke-test creator skipped.
